# Personal Loan Approval Prediction — Executed Training Notebook

## Objective
Predict **lender approval probability** (`loan_approved=1`) from the personal loan application attributes.

This notebook was executed against `personal_loans.csv`. It performs EDA, leakage-aware preprocessing, cross-validated model comparison, hyperparameter tuning, held-out testing, probability calibration and model export.

**Data caveat:** The dataset is synthetic, with class proportions and target patterns generated for coursework. Reported metrics measure performance on this synthetic generator only; they are not evidence of real-world lending accuracy.

## 1. Imports and reproducibility
Fixed random seeds make the split and model comparisons repeatable. All preprocessing is placed inside the estimator pipeline to prevent train/test leakage.

In [3]:
import warnings, os, json, joblib
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, brier_score_loss, classification_report, ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay
from sklearn.calibration import CalibratedClassifierCV, CalibrationDisplay
from IPython.display import display
RANDOM_STATE=42
DATA_PATH="personal_loans.csv"


## 2. Load and audit the data
Check dimensions, duplicates, missingness, dtypes and class balance before training.

In [5]:
df=pd.read_csv(DATA_PATH)
print("Shape:",df.shape," | Duplicate rows:",df.duplicated().sum())
display(df.head())
display(df.dtypes.astype(str).value_counts().rename("column_count").to_frame())
display(df.isna().sum().sort_values(ascending=False).head(15).to_frame("missing"))
print("Target proportions:")
display(df["loan_approved"].value_counts(normalize=True).sort_index().rename(index={0:"Rejected",1:"Approved"}).to_frame("share"))


Shape: (1000, 29)  | Duplicate rows: 0
Target proportions:


## 3. Exploratory data analysis
Histograms expose skew and unusual ranges; the target chart shows class balance. We retain unusual values unless data validation identifies an actual error.

In [7]:
df["loan_approved"].value_counts().sort_index().plot(kind="bar",rot=0,title="Target count (0=Rejected, 1=Approved)")
plt.xlabel("loan_approved"); plt.ylabel("Applications"); plt.show()
numeric_cols=df.select_dtypes(include=np.number).columns.tolist()
df[numeric_cols].hist(figsize=(15,10),bins=25)
plt.suptitle("Numeric feature distributions"); plt.tight_layout(); plt.show()
if "product_name" in df:
    display(df.groupby("product_name")["loan_approved"].agg(applications="count",approval_rate="mean"))


## 4. Define features and target
Remove identifiers because they are arbitrary labels and can encourage memorization. Keep all remaining available application/product attributes. The target is separated before any preprocessing.

In [9]:
TARGET="loan_approved"
id_cols=[c for c in ["applicant_id","application_id"] if c in df.columns]
X=df.drop(columns=[TARGET]+id_cols)
y=df[TARGET].astype(int)
assert set(y.unique())=={0,1}
print("Rows:",len(X),"Predictors:",X.shape[1])


Rows: 1000 Predictors: 26


## 5. Stratified split and preprocessing
Hold out 20% for the final evaluation. Numeric columns are median-imputed and standardized; categorical columns are mode-imputed and one-hot encoded. Unknown categories at inference are safely ignored.

In [11]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.20,random_state=42,stratify=y)
num_features=X_train.select_dtypes(include=np.number).columns.tolist()
cat_features=X_train.select_dtypes(exclude=np.number).columns.tolist()
try:
    ohe=OneHotEncoder(handle_unknown="ignore",sparse_output=False)
except TypeError:
    ohe=OneHotEncoder(handle_unknown="ignore",sparse=False)
numeric_pipe=Pipeline([("imputer",SimpleImputer(strategy="median")),("scaler",StandardScaler())])
categorical_pipe=Pipeline([("imputer",SimpleImputer(strategy="most_frequent")),("onehot",ohe)])
preprocessor=ColumnTransformer([("num",numeric_pipe,num_features),("cat",categorical_pipe,cat_features)])
print("Train:",len(X_train),"Test:",len(X_test),"Numeric:",len(num_features),"Categorical:",len(cat_features))


Train: 800 Test: 200 Numeric: 21 Categorical: 5


## 6. Cross-validated model comparison
Compare a dummy baseline, Logistic Regression, Random Forest, Extra Trees and HistGradientBoosting. Three-fold stratified CV is used to keep the run practical for a 1,000-row coursework dataset. ROC-AUC is the primary ranking metric; PR-AUC and F1 provide additional perspectives.

In [13]:
cv=StratifiedKFold(n_splits=3,shuffle=True,random_state=42)
estimators={
 "Dummy baseline":DummyClassifier(strategy="most_frequent"),
 "Logistic Regression":LogisticRegression(max_iter=2000,class_weight="balanced",C=1.0),
 "Random Forest":RandomForestClassifier(n_estimators=180,min_samples_leaf=2,max_features="sqrt",class_weight="balanced",random_state=42,n_jobs=-1),
 "Extra Trees":ExtraTreesClassifier(n_estimators=180,min_samples_leaf=2,max_features="sqrt",class_weight="balanced",random_state=42,n_jobs=-1),
 "Hist Gradient Boosting":HistGradientBoostingClassifier(max_iter=120,learning_rate=0.08,l2_regularization=1.0,random_state=42)
}
pipelines={}
comparison=[]
for name,est in estimators.items():
    pipe=Pipeline([("preprocess",preprocessor),("model",est)])
    scores=cross_validate(pipe,X_train,y_train,cv=cv,scoring={"roc_auc":"roc_auc","pr_auc":"average_precision","f1":"f1"},n_jobs=1)
    pipelines[name]=pipe
    comparison.append({"model":name,"cv_roc_auc_mean":scores["test_roc_auc"].mean(),"cv_roc_auc_std":scores["test_roc_auc"].std(),"cv_pr_auc_mean":scores["test_pr_auc"].mean(),"cv_f1_mean":scores["test_f1"].mean()})
comparison_df=pd.DataFrame(comparison).sort_values("cv_roc_auc_mean",ascending=False).reset_index(drop=True)
display(comparison_df.round(4))


## 7. Hyperparameter tuning
Tune Random Forest and Extra Trees with randomized search. The compact search tests tree depth, leaf size, split size, feature sampling and estimator count. All selection occurs using training folds only.

In [15]:
param_space={
 "model__n_estimators":[120,200,300],
 "model__max_depth":[None,6,10,16],
 "model__min_samples_split":[2,5,10],
 "model__min_samples_leaf":[1,2,4],
 "model__max_features":["sqrt",0.6,0.8]
}
tuned={}
for name,est in {
 "Random Forest":RandomForestClassifier(class_weight="balanced",random_state=42,n_jobs=-1),
 "Extra Trees":ExtraTreesClassifier(class_weight="balanced",random_state=42,n_jobs=-1)
}.items():
    pipe=Pipeline([("preprocess",preprocessor),("model",est)])
    search=RandomizedSearchCV(pipe,param_space,n_iter=5,scoring="roc_auc",cv=cv,random_state=42,n_jobs=1,refit=True)
    search.fit(X_train,y_train)
    tuned[name]=search
    print(name,"best CV ROC-AUC:",round(search.best_score_,4),"best params:",search.best_params_)


Random Forest best CV ROC-AUC: 0.8899 best params: {'model__n_estimators': 120, 'model__min_samples_split': 10, 'model__min_samples_leaf': 4, 'model__max_features': 0.6, 'model__max_depth': 6}
Extra Trees best CV ROC-AUC: 0.8709 best params: {'model__n_estimators': 300, 'model__min_samples_split': 2, 'model__min_samples_leaf': 1, 'model__max_features': 0.8, 'model__max_depth': 6}


## 8. Select the best model and evaluate on the untouched test set
Selection uses cross-validation ROC-AUC. Only after selection is the chosen pipeline fitted and evaluated on the held-out test partition. Threshold metrics use 0.50 as a transparent default, not a universal lending threshold.

In [17]:
candidate_models=dict(pipelines)
candidate_scores=dict(zip(comparison_df["model"],comparison_df["cv_roc_auc_mean"]))
for name,search in tuned.items():
    candidate_models[name+" (tuned)"]=search.best_estimator_
    candidate_scores[name+" (tuned)"]=search.best_score_
best_name=max(candidate_scores,key=candidate_scores.get)
best_model=candidate_models[best_name]
best_model.fit(X_train,y_train)
y_pred=best_model.predict(X_test)
y_prob=best_model.predict_proba(X_test)[:,1]
test_metrics={
 "selected_model":best_name,
 "test_accuracy":accuracy_score(y_test,y_pred),
 "test_precision":precision_score(y_test,y_pred,zero_division=0),
 "test_recall":recall_score(y_test,y_pred,zero_division=0),
 "test_f1":f1_score(y_test,y_pred,zero_division=0),
 "test_roc_auc":roc_auc_score(y_test,y_prob),
 "test_pr_auc":average_precision_score(y_test,y_prob),
 "test_brier_score":brier_score_loss(y_test,y_prob)
}
display(pd.Series(test_metrics,name="value").to_frame())
print(classification_report(y_test,y_pred,target_names=["Rejected","Approved"],zero_division=0))


              precision    recall  f1-score   support

    Rejected       0.73      0.68      0.71        90
    Approved       0.75      0.80      0.78       110

    accuracy                           0.74       200
   macro avg       0.74      0.74      0.74       200
weighted avg       0.74      0.74      0.74       200



## 9. Test-set diagnostics
The confusion matrix shows threshold errors. ROC and precision-recall curves evaluate ranking across thresholds. Inspect false approvals and false rejections in the context of the project's intended use.

In [19]:
ConfusionMatrixDisplay.from_predictions(y_test,y_pred,display_labels=["Rejected","Approved"])
plt.title("Held-out test confusion matrix"); plt.show()
RocCurveDisplay.from_predictions(y_test,y_prob); plt.title("Held-out test ROC curve"); plt.show()
PrecisionRecallDisplay.from_predictions(y_test,y_prob); plt.title("Held-out test precision-recall curve"); plt.show()


## 10. Calibrate the probability output
Use sigmoid calibration with three-fold CV on training data only. Brier score evaluates probability error (lower is better). Calibration is necessary because the requested output is an estimated probability rather than only a class label.

In [21]:
try:
    calibrated_model=CalibratedClassifierCV(estimator=best_model,method="sigmoid",cv=3)
except TypeError:
    calibrated_model=CalibratedClassifierCV(base_estimator=best_model,method="sigmoid",cv=3)
calibrated_model.fit(X_train,y_train)
calibrated_prob=calibrated_model.predict_proba(X_test)[:,1]
print("Uncalibrated Brier:",round(brier_score_loss(y_test,y_prob),4))
print("Calibrated Brier:",round(brier_score_loss(y_test,calibrated_prob),4))
CalibrationDisplay.from_predictions(y_test,calibrated_prob,n_bins=8,strategy="quantile")
plt.title("Held-out test calibration curve"); plt.show()


Uncalibrated Brier: 0.1616
Calibrated Brier: 0.1687


## 11. Example prediction for a new application
The model expects the same feature columns as training. This demonstration uses one held-out row solely to illustrate the interface. Replace it with a new applicant's values in an application.

In [23]:
example_application=X_test.iloc[[0]].copy()
approval_probability=calibrated_model.predict_proba(example_application)[0,1]
print(f"Estimated lender approval probability: {approval_probability:.1%}")
print("Predicted label at 0.50 threshold:", "Approved" if approval_probability>=0.50 else "Rejected")
display(example_application)


Estimated lender approval probability: 87.0%
Predicted label at 0.50 threshold: Approved


## 12. Save the trained model and metrics
The saved calibrated pipeline includes imputers, encoders, scaler and model. Keep the same scikit-learn version when loading the artifact.

In [25]:
MODEL_FILE="personal_loan_approval_model.joblib"
METRICS_FILE="personal_model_metrics.json"
joblib.dump(calibrated_model,MODEL_FILE)
with open(METRICS_FILE,"w") as f:
    json.dump({"loan_type":"personal","selected_model":best_name,"metrics":test_metrics,"cv_comparison":comparison_df.to_dict(orient="records")},f,indent=2)
print("Saved model:",MODEL_FILE)
print("Saved metrics:",METRICS_FILE)


Saved model: personal_loan_approval_model.joblib
Saved metrics: personal_model_metrics.json


## 13. Interpretation and limitations
- These are synthetic data and synthetic labels; test metrics only measure recovery of the simulated patterns.
- A probability score is not a guarantee of lender approval.
- Real deployment requires representative historical outcomes, out-of-time validation, calibration monitoring, subgroup fairness checks, explainability, privacy controls and compliance review.
- Avoid using the output as the sole basis for an actual credit decision.